In [ ]:
import pandas as pd

df = pd.read_csv("processed_dataset_final.csv")

print(df.shape)
df.head()

(1459153, 7)


,lang,country,reconstructed_text,main_sentiment,sentiment_score,main_stance,stance_score
0,cs,Slovakia,Weekend selection: Zelensky was not prepared f...,neutral,0.5044,Unsure,0.8667
1,ro,Italy,"The Ukrainian war, Charles Michel of Kiev, the...",neutral,0.9146,Unsure,0.4485
2,te,India,The invention of the Shark drone is a new chap...,neutral,0.5799,Unsure,0.8056
3,te,India,Will nuclear war be over? Those countries that...,negative,0.7900,Unsure,0.2205
4,te,India,Boys fight: Students fight in coaching center ...,negative,0.6543,Pro Ukraine,0.9286


In [ ]:
df.isnull().sum()

lang                   0
country                0
reconstructed_text    24
main_sentiment         0
sentiment_score        0
main_stance            0
stance_score           0
dtype: int64

In [ ]:
df = df.dropna(subset=["reconstructed_text"])

print(df.shape)

(1459129, 7)


In [ ]:
df.isnull().sum()

lang                  0
country               0
reconstructed_text    0
main_sentiment        0
sentiment_score       0
main_stance           0
stance_score          0
dtype: int64

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df['reconstructed_text'].duplicated().sum()

np.int64(16567)

In [ ]:
df.to_csv('checkpoint.csv')

In [ ]:
entity_count = (
    df["reconstructed_text"]
    .fillna("")
    .str.contains(r"&[a-zA-Z]+;", regex=True)
    .sum()
)

print("Rows containing HTML entities:", entity_count)

Rows containing HTML entities: 9489


In [ ]:
import re
from collections import Counter

entities = Counter()

for text in df["reconstructed_text"].fillna(""):
    entities.update(re.findall(r"&[a-zA-Z]+;", text))

print(entities.most_common(20))

[('&amp;', 8980), ('&gt;', 2961), ('&lt;', 39)]


In [ ]:
import html

def unescape_text(x):
    if isinstance(x, str):
        return html.unescape(x)
    return x

df["reconstructed_text"] = df["reconstructed_text"].apply(unescape_text)

In [ ]:
entity_count = (
    df["reconstructed_text"]
    .fillna("")
    .str.contains(r"&[a-zA-Z]+;", regex=True)
    .sum()
)

print(entity_count)

1


In [ ]:
import re
for text in df["reconstructed_text"].dropna():
    if re.search(r"&[a-zA-Z]+;", text):
        print(text)
        break

RT: Operation Support # Ukraine ️ Tuesday 8 March at 12 noon ➡️ &amp;amp;amp;amp;amp;amp;amp;amp;amp;amp ;amp; amp; amp; amp; amp; 200 French private radio stations join forces with a ...


In [ ]:
from collections import Counter
url_patterns = Counter()

for text in df["reconstructed_text"].dropna():
    matches = re.findall(
        r"(https?://\S+|www\.\S+|<\s*URL[^>]*\s*>)",
        text,
        flags=re.IGNORECASE
    )
    url_patterns.update(matches[:1])

print("Rows with URLs:", sum(url_patterns.values()))
print(url_patterns.most_common(20))

Rows with URLs: 489461
[('< URL_1 >', 105795), ('https://t.co/X3flQUBL0r', 740), ('https://t.co/AXC5qRcEPB', 658), ('https://t.co/Wa58gYGZwF', 544), ('https://t.co/AXC5qRugeb', 387), ('https://t.co/AXC5qRuO3J', 351), ('https://t.co/dm7SyBIpO6', 303), ('https://t.co/gx4btIcPZF', 185), ('https://t.c…', 181), ('https://t.co…', 173), ('https://t.co/…', 168), ('https://t.co/xkYNpPFlF9', 161), ('https://t…', 161), ('https://t.co/LO1PjaYX9U', 155), ('https://t.co/E5MBaNCDfC', 148), ('https://t.…', 143), ('https://t.co/NEDMP2uP6W', 141), ('https://t.co/cz5NTchyMw', 138), ('https://t.co/X3flQUk9BR', 132), ('https://t.co/xkYNpPXuTh', 115)]


In [ ]:
def remove_urls(text):
    if not isinstance(text, str):
        return text

    return re.sub(
        r'https?://\S+|www\.\S+|<\s*URL[^>]*\s*>',
        '',
        text,
        flags=re.IGNORECASE
    )

df["reconstructed_text"] = df["reconstructed_text"].apply(remove_urls)

In [ ]:
url_count = df["reconstructed_text"].str.contains(
    r'https?://|www\.|<\s*URL[^>]*\s*>',
    regex=True,
    case=False,
    na=False
).sum()

print("Rows still containing URLs:", url_count)

Rows still containing URLs: 10


In [ ]:
remaining = df[
    df["reconstructed_text"].str.contains(
        r'https?://|www\.|<\s*URL[^>]*\s*>',
        regex=True,
        na=False
    )
]

for text in remaining["reconstructed_text"]:
    print(text)
    print("-" * 100)

Black Sea explosion on Russian warship. # Kiev claims, for # Moscow it is an accident. ://www.
----------------------------------------------------------------------------------------------------
Russia attacks Ukrainian civilians at bus stop kills three people Watch the news on World News with Prima Alvernia at 11:59 pm only on tvOne & live streaming at http:// http:// #KabarDuniatvOne #CariBeritaditvOne
----------------------------------------------------------------------------------------------------
Russian Su-30 fighter jet destroys Ukrainian Snake Island Watch the news on the World News with Prima Alverina at 11:59 pm only on tvOne & live streaming at http:// http:// #KabarDuniatvOne #CariBeritaditvOne
----------------------------------------------------------------------------------------------------
Russian attack hits apartment in Odessa, Ukraine Watch the news on the World News with Rendra Kusuma at 2400 WIB only on tvOne & live streaming at http:// http:// #KabarDuniatvOne 

In [ ]:
import re

def remove_url_fragments(text):
    if not isinstance(text, str):
        return text

    text = re.sub(r'://\S*', '', text)
    text = re.sub(r'http\S*', '', text)
    text = re.sub(r'www\.\S*', '', text)

    return text

df["reconstructed_text"] = df["reconstructed_text"].apply(remove_url_fragments)

In [ ]:
df["reconstructed_text"].str.contains(
    r'https?://|://|www\.|http',
    regex=True,
    na=False
).sum()

np.int64(0)

In [ ]:
rt_count = df["reconstructed_text"].str.contains(
    r'^RT\b',
    regex=True,
    case=False,
    na=False
).sum()

print("Rows starting with RT:", rt_count)

Rows starting with RT: 106449


In [ ]:
import re

def remove_rt(text):
    if not isinstance(text, str):
        return text

    return re.sub(
        r'^RT\s*:?\s*',
        '',
        text,
        flags=re.IGNORECASE
    )

df["reconstructed_text"] = df["reconstructed_text"].apply(remove_rt)

In [ ]:
remaining_rt = df["reconstructed_text"].str.contains(
    r'^RT\b',
    regex=True,
    case=False,
    na=False
).sum()

print("Rows still starting with RT:", remaining_rt)

Rows still starting with RT: 0


In [ ]:
mentions = df["reconstructed_text"].str.contains(
    r'@\s*\w+',
    regex=True,
    na=False
).sum()

print("Rows containing mentions:", mentions)

Rows containing mentions: 81265


In [ ]:
def remove_mentions(text):
    if not isinstance(text, str):
        return text

    return re.sub(r'@\s*\w+', '', text)

df["reconstructed_text"] = df["reconstructed_text"].apply(remove_mentions)

In [ ]:
remaining = df["reconstructed_text"].str.contains(
    r'@\s*\w+',
    regex=True,
    na=False
).sum()

print("Remaining mentions:", remaining)

Remaining mentions: 0


In [ ]:
total_hashtags = df["reconstructed_text"].str.contains(
    r'#\s*\w+',
    regex=True,
    na=False
).sum()

print("Rows containing hashtags:", total_hashtags)

Rows containing hashtags: 409816


In [ ]:
hashtag_rows = df[
    df["reconstructed_text"].str.contains(
        r'#\s*\w+',
        regex=True,
        na=False
    )
]

for text in hashtag_rows["reconstructed_text"].sample(20):
    print(text)
    print("-"*100)

# Nato is scaring the optimistic: The #war in # Ukraine could last for years # stoltenberg # Russia # June19 # time daily
----------------------------------------------------------------------------------------------------
# PolandGoodDay. # Russia 's next phase of operation is the takeover of Donbass - # we invite you to #program ➡️ # watch # YouTube channel #turn on the truth # TVRepublika
----------------------------------------------------------------------------------------------------
The visit to # Irpin, a devastated city that has become a symbol of the atrocities committed by the Russian army, " emotionized " # EmmanuelMacron, the Indian ...
----------------------------------------------------------------------------------------------------
The # Kremlin said explosions that rocked Kyiv on Monday were part of what it calls its " special #military operation. " 
----------------------------------------------------------------------------------------------------
# Urgent: Emergen

In [ ]:
def split_hashtags(text):
    if not isinstance(text, str):
        return text

    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)

    text = re.sub(r'_', ' ', text)

    text = re.sub(r'(?<=\w)-(?=\w)', ' ', text)

    text = re.sub(r'#', '', text)

    return text

df["reconstructed_text"] = df["reconstructed_text"].apply(split_hashtags)

In [ ]:
remaining_hashes = df["reconstructed_text"].str.contains(
    r'#',
    regex=True,
    na=False
).sum()

print("Remaining # symbols:", remaining_hashes)

Remaining # symbols: 0


In [ ]:
import re

emoji_pattern = re.compile(
    "["
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F900-\U0001F9FF"
    "\U00002600-\U000027BF"
    "]"
)

emoji_count = df["reconstructed_text"].apply(
    lambda x: bool(emoji_pattern.search(x)) if isinstance(x, str) else False
).sum()

print("Rows containing emojis:", emoji_count)

Rows containing emojis: 96689


In [ ]:
from collections import Counter

emoji_counter = Counter()

for text in df["reconstructed_text"].dropna():
    emojis = emoji_pattern.findall(text)
    emoji_counter.update(emojis)

print(emoji_counter.most_common(30))

[('🔴', 37764), ('⚡', 14625), ('➡', 13169), ('👉', 12439), ('👇', 8427), ('📌', 4504), ('✍', 4479), ('📺', 2925), ('🏻', 2056), ('📷', 2015), ('🔹', 1812), ('📸', 1250), ('❗', 1053), ('🗣', 1042), ('🚨', 1031), ('🎧', 562), ('🏼', 557), ('♦', 506), ('📰', 500), ('✅', 463), ('🔊', 414), ('🔺', 359), ('📹', 347), ('🎙', 329), ('🖥', 309), ('🧵', 304), ('🚀', 293), ('❓', 286), ('💭', 275), ('📱', 259)]


In [ ]:
emoji_pattern = re.compile(
    "["
    "\U0001F300-\U0001F5FF"  # Miscellaneous Symbols and Pictographs
    "\U0001F600-\U0001F64F"  # Emoticons
    "\U0001F680-\U0001F6FF"  # Transport and Map Symbols
    "\U0001F700-\U0001F77F"  # Alchemical Symbols
    "\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
    "\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
    "\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
    "\U0001FA00-\U0001FAFF"  # Chess Symbols, Extended-A
    "\U00002600-\U000027BF"  # Miscellaneous Symbols
    "]+",
    flags=re.UNICODE
)

def remove_emojis(text):
    if not isinstance(text, str):
        return text

    return emoji_pattern.sub('', text)

df["reconstructed_text"] = df["reconstructed_text"].apply(remove_emojis)

In [ ]:
emoji_remaining = df["reconstructed_text"].apply(
    lambda x: bool(emoji_pattern.search(x)) if isinstance(x, str) else False
).sum()

print("Rows still containing emojis:", emoji_remaining)

Rows still containing emojis: 0


In [ ]:
broken_apostrophes = df["reconstructed_text"].str.count(
    r"\s+'|'\s+"
).sum()

print("Broken apostrophe occurrences:", broken_apostrophes)

Broken apostrophe occurrences: 383556


In [ ]:
patterns = Counter()

for text in df["reconstructed_text"].dropna():
    matches = re.findall(r"\w+\s*'\s*\w+", text)
    patterns.update(matches)

print(patterns.most_common(30))

[("Russia 's", 60599), ("Ukraine 's", 39028), ("Putin 's", 21625), ("n't", 15405), ("Putin's", 7711), ("country 's", 6965), ("Moscow 's", 6071), ("It 's", 4912), ("it 's", 4260), ("Zelensky 's", 3502), ("Europe 's", 2866), ("Here 's", 2743), ("Biden 's", 2707), ("world 's", 2701), ("China 's", 2618), ("can't", 1985), ("Kiev 's", 1799), ("I 'm", 1756), ("today 's", 1663), ("he 's", 1563), ("People 's", 1549), ("Today 's", 1490), ("NATO 's", 1462), ("Kyiv 's", 1417), ("India 's", 1391), ("We 're", 1385), ("city 's", 1371), ("Germany 's", 1363), ("president 's", 1331), ("that 's", 1314)]


In [ ]:
def fix_apostrophes(text):
    if not isinstance(text, str):
        return text

    return re.sub(r"\s*'\s*", "'", text)

df["reconstructed_text"] = df["reconstructed_text"].apply(fix_apostrophes)

In [ ]:
remaining = df["reconstructed_text"].str.contains(
    r"\s+'|'\s+",
    regex=True,
    na=False
).sum()

print("Remaining broken apostrophes:", remaining)

Remaining broken apostrophes: 0


In [ ]:
chat_words = [
    "u", "ur", "urs", "r", "y", "ya",
    "btw", "imo", "imho", "idk", "ikr",
    "lol", "lmao", "rofl", "omg", "wtf",
    "thx", "ty", "pls", "plz", "gr8",
    "b4", "bc", "cuz", "coz", "tho",
    "tmrw", "asap", "fyi", "afaik"
]
chat_counts = Counter()

for text in df["reconstructed_text"].dropna():
    words = re.findall(r"\b\w+\b", text.lower())

    for word in words:
        if word in chat_words:
            chat_counts[word] += 1

print(chat_counts.most_common(30))

[('u', 32496), ('r', 1206), ('y', 174), ('ur', 130), ('ya', 57), ('bc', 56), ('asap', 33), ('tho', 13), ('fyi', 9), ('imo', 6), ('btw', 6), ('pls', 5), ('imho', 3), ('lol', 3), ('urs', 2), ('b4', 2), ('ty', 2), ('wtf', 1), ('thx', 1), ('lmao', 1), ('omg', 1)]


In [ ]:
broken_contractions = df["reconstructed_text"].str.contains(
    r"\b\w+\s+n't\b",
    regex=True,
    na=False
).sum()

print("Broken contractions:", broken_contractions)

Broken contractions: 14584


In [ ]:
mask = df["reconstructed_text"].str.contains(
    r"\b\w+\s+n't\b",
    regex=True,
    na=False
)

for idx, text in df.loc[mask, "reconstructed_text"].head(50).items():
    print("Index:", idx)
    print(text)
    print("-" * 80)

Index: 133
'Listen to our voice. The people of Ukraine want peace. The Ukrainian authorities want peace. We do n't want war,'
--------------------------------------------------------------------------------
Index: 150
The war is not over. Do n't rest, Putin's goal is destruction!: Ukraine key comments
--------------------------------------------------------------------------------
Index: 164
I did n't take a gap .. I just came! Elon Musk would n't stop if there was a war! < URL 
--------------------------------------------------------------------------------
Index: 196
Locals describe how war is affecting Transcarpathia: Without electricity we wo n't flood
--------------------------------------------------------------------------------
Index: 226
Czech soldiers in Ukraine: We are counting on death. We do n't lack willpower, the war may last for years
--------------------------------------------------------------------------------
Index: 228
Why the war wo n't just end. Analyst describe

In [ ]:
def fix_broken_contractions(text):
    if not isinstance(text, str):
        return text

    text = re.sub(r"\b(\w+)\s+n't\b", r"\1n't", text)

    return text

df["reconstructed_text"] = df["reconstructed_text"].apply(fix_broken_contractions)

In [ ]:
remaining = df["reconstructed_text"].str.contains(
    r"\b\w+\s+n't\b",
    regex=True,
    na=False
).sum()

print("Remaining broken contractions:", remaining)

Remaining broken contractions: 0


In [ ]:
df["reconstructed_text"] = df["reconstructed_text"].str.lower()

In [ ]:
from collections import Counter
import string

punct_counts = Counter()

for text in df["reconstructed_text"].dropna():
    for char in text:
        if char in string.punctuation:
            punct_counts[char] += 1

print(punct_counts.most_common(30))

[('.', 1110645), (',', 952675), (':', 484668), ("'", 420377), ('"', 382646), ('-', 241512), ('?', 55472), ('<', 41224), ('>', 39850), ('|', 32260), ('(', 31717), (')', 31553), ('$', 16312), ('!', 15464), (';', 14367), ('/', 11188), ('%', 9610), ('+', 9380), ('&', 9152), ('[', 7808), (']', 7675), ('*', 2579), ('@', 227), ('`', 165), ('=', 84), ('~', 40), ('\\', 11), ('^', 7), ('}', 2), ('{', 1)]


In [ ]:
import string

def remove_punctuation(text):
    if not isinstance(text, str):
        return text

    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    return text

df["reconstructed_text"] = df["reconstructed_text"].apply(remove_punctuation)

In [ ]:
def normalize_whitespace(text):
    if not isinstance(text, str):
        return text

    return re.sub(r'\s+', ' ', text).strip()

df["reconstructed_text"] = df["reconstructed_text"].apply(normalize_whitespace)

In [ ]:
df['reconstructed_text'].iloc[223471:223482]

223471    how do i tell my daughter about the war ukrain...
223472    as part of our ukraine coverage user mention 1...
223473    here is a round up of the latest news on the w...
223474    the numbers signing up from abroad to fight in...
223475    opinion while ukrainians fight for their lives...
223476    the number of people signing up from abroad to...
223477    yandex considered russia s google shaken and i...
223478    ukrainians are making and sharing recordings o...
223479    inflation has hit its highest level in decades...
223480    since the beginning of this year sentiment has...
223481    an opinion poll in sweden showed that for the ...
Name: reconstructed_text, dtype: object

In [ ]:
df.to_csv(
    "dataset_cleaned.csv",
    index=False,
    encoding="utf-8"
)

In [ ]:
df.isnull().sum()

lang                  0
country               0
reconstructed_text    0
main_sentiment        0
sentiment_score       0
main_stance           0
stance_score          0
dtype: int64

In [ ]:
df['reconstructed_text'].iloc[1402683]

''